# Brussels residents population generation
First, we generate the synthetic population of residents of the Brussels Capital Region (BCR) from the 2021 Census data with the following attributes:

* sex
* age (M - 5yr brackets)
* education
* employment (L - INAC, EMP, UNE)
* statistical sector

Input files: S01, S11, S12, HC04_3, HC04_1, HC23_2

Next, we filter to only get the employed population and add the following attributes:
* professional status (SAL, SELF_NS, SELF_S, EMP_OTH) = employement type
* industry = work sector

In [ ]:
import numpy as np
import pandas as pd
from ipfn.ipfn import ipfn as IPFN
import re

rng = np.random.default_rng(42)

# 1 Brussels population generation

In [ ]:
# Load data
age5_sector = pd.read_csv("input_data/TF_CENSUS_2021_S01_cleaned.csv")
edu_sector = pd.read_csv("input_data/TF_CENSUS_2021_S11_cleaned.csv")
emp_sector = pd.read_csv("input_data/TF_CENSUS_2021_S12_cleaned.csv")
prov_age5_edu = pd.read_csv("input_data/TF_CENSUS_2021_HC04_3_cleaned.csv")
prov_age5_emp = pd.read_csv("input_data/TF_CENSUS_2021_HC04_1_cleaned.csv")
prov_age5_edu_emp = pd.read_csv("input_data/TF_CENSUS_2021_HC23_2_cleaned.csv")

In [ ]:
# Define categories and index maps
sexes = sorted(prov_age5_edu["CD_SEX"].unique())
age5 = sorted(prov_age5_edu["CD_AGE"].unique())
educations = sorted(prov_age5_edu["CD_EDU"].unique())
employments = sorted(prov_age5_emp["CD_EMPLOYMENT"].unique())
sectors = sorted(age5_sector["CD_SECTOR"].unique())

S = len(sexes)
A5 = len(age5)
E = len(educations)
M = len(employments)
C = len(sectors)

# index maps
sex_i = {v: i for i, v in enumerate(sexes)}
age5_i = {v: i for i, v in enumerate(age5)}
edu_i = {v: i for i, v in enumerate(educations)}
emp_i = {v: i for i, v in enumerate(employments)}
sec_i = {v: i for i, v in enumerate(sectors)}

In [ ]:
# Build margin arrays
# Province: sex x age5 x education
T_age5_edu = np.zeros((S, A5, E))
for _, r in prov_age5_edu.iterrows():
    T_age5_edu[sex_i[r["CD_SEX"]], age5_i[r["CD_AGE"]], edu_i[r["CD_EDU"]]] = r[
        "MS_POP"
    ]

# Province: sex x age5 x employment
T_age5_emp = np.zeros((S, A5, M))
for _, r in prov_age5_emp.iterrows():
    T_age5_emp[sex_i[r["CD_SEX"]], age5_i[r["CD_AGE"]], emp_i[r["CD_EMPLOYMENT"]]] = r[
        "MS_POP"
    ]

# Province: sex x age5 x education x employment
T_age5_edu_emp = np.zeros((S, A5, E, M))
for _, r in prov_age5_edu_emp.iterrows():
    T_age5_edu_emp[
        sex_i[r["CD_SEX"]],
        age5_i[r["CD_AGE"]],
        edu_i[r["CD_EDU"]],
        emp_i[r["CD_EMPLOYMENT"]],
    ] = r["MS_POP"]

# Sector constraints
T_age5_sec = np.zeros((S, A5, C))
for _, r in age5_sector.iterrows():
    T_age5_sec[sex_i[r["CD_SEX"]], age5_i[r["CD_AGE"]], sec_i[r["CD_SECTOR"]]] = r[
        "MS_POP"
    ]

T_edu_sec = np.zeros((S, E, C))
for _, r in edu_sector.iterrows():
    T_edu_sec[sex_i[r["CD_SEX"]], edu_i[r["CD_EDU"]], sec_i[r["CD_SECTOR"]]] = r[
        "MS_POP"
    ]
T_emp_sec = np.zeros((S, M, C))
for _, r in emp_sector.iterrows():
    T_emp_sec[sex_i[r["CD_SEX"]], emp_i[r["CD_EMPLOYMENT"]], sec_i[r["CD_SECTOR"]]] = r[
        "MS_POP"
    ]

In [ ]:
# Convergence checker
def check_ipf_convergence(result_s, s_idx, tol=1e-2):
    checks = {
        "age5-edu-emp": (
            result_s.sum(axis=3),
            T_age5_edu_emp[s_idx],
        ),
        "age5-edu": (
            result_s.sum(axis=(2, 3)),
            T_age5_edu[s_idx],
        ),
        "age5-emp": (
            result_s.sum(axis=(1, 3)),
            T_age5_emp[s_idx],
        ),
        "age5-sector": (
            result_s.sum(axis=(1, 2)),
            T_age5_sec[s_idx],
        ),
        "edu-sector": (
            result_s.sum(axis=(0, 2)),
            T_edu_sec[s_idx],
        ),
        "emp-sector": (
            result_s.sum(axis=(0, 1)),
            T_emp_sec[s_idx],
        ),
    }

    max_err = 0.0
    for name, (fit, target) in checks.items():
        err = np.max(np.abs(fit - target))
        max_err = max(max_err, err)
        print(f"{name}: max abs error = {err:.4f}")

    if max_err <= tol:
        print(f"Convergence check: OK (max error {max_err:.4f} <= {tol})")
    else:
        print(f"Convergence check: NOT OK (max error {max_err:.4f} > {tol})")

    return max_err

In [ ]:
# Run IPF separately for each gender
result = np.zeros((S, A5, E, M, C), dtype=float)

for s_idx, sex in enumerate(sexes):
    print(f"Running IPF for sex = {sex}")

    # Sex-specific margins
    aggregates = [
        T_age5_edu_emp[s_idx],  # age5 x edu x emp
        T_age5_edu[s_idx],      # age5 x edu
        T_age5_emp[s_idx],      # age5 x emp
        T_age5_sec[s_idx],      # age5 x sector
        T_edu_sec[s_idx],       # edu x sector
        T_emp_sec[s_idx],       # emp x sector
    ]

    dimensions = [
        [0, 1, 2],  # age5-edu-emp
        [0, 1],     # age5-edu
        [0, 2],     # age5-emp
        [0, 3],     # age5-sector
        [1, 3],     # edu-sector
        [2, 3],     # emp-sector
    ]

    # Sex-specific seed based on uniform distribution
    total_pop_sex = T_age5_edu[s_idx].sum()
    seed = np.ones((A5, E, M, C), dtype=float) * (total_pop_sex / (A5 * E * M * C))

    ipf = IPFN(seed, aggregates, dimensions, max_iteration=500, convergence_rate=1e-5)
    result_s = ipf.iteration()

    print(f"Result shape: {result_s.shape}")
    print(f"Total population: {result_s.sum():.0f} / {total_pop_sex:.0f}")

    check_ipf_convergence(result_s, s_idx, tol=1e-2)

    result[s_idx] = result_s

print("\nFinished sex-specific IPF.")
print(f"Total population in result: {result.sum():.0f}")
print(f"Expected total: {T_age5_edu.sum():.0f}")

In [ ]:
# Round results and adjust to match totals
result_rounded = np.zeros_like(result, dtype=int)

for s_idx, sex in enumerate(sexes):
    slice_result = result[s_idx]
    slice_rounded = np.round(slice_result)
    target_total = int(T_age5_edu[s_idx].sum())
    total_rounded = int(slice_rounded.sum())

    print(f"\nSex {sex}:")
    print(f"After rounding: {total_rounded}")

    if total_rounded != target_total:
        diff = target_total - total_rounded
        fractional = slice_result - slice_rounded

        flat_fractional = fractional.flatten()
        flat_rounded = slice_rounded.flatten()

        if diff > 0:
            mask = flat_rounded >= 0
            indices = np.where(mask)[0]
            sorted_indices = indices[np.argsort(-flat_fractional[indices])][:diff]
            for idx in sorted_indices:
                flat_rounded[idx] += 1
        else:
            mask = flat_rounded > 0
            indices = np.where(mask)[0]
            sorted_indices = indices[np.argsort(flat_fractional[indices])][: abs(diff)]
            for idx in sorted_indices:
                flat_rounded[idx] -= 1

        slice_rounded = flat_rounded.reshape(slice_result.shape)

    result_rounded[s_idx] = slice_rounded.astype(int)

    print(f"After adjustment: {slice_rounded.sum():.0f}")
    print(f"Target total:    {target_total}")

print(f"\nAfter adjustment: {result_rounded.sum():.0f}")

In [ ]:
# Save results
# Convert to DataFrame for easier use
rows = []
for s_idx, sex in enumerate(sexes):
    for a5_idx, age in enumerate(age5):
        for e_idx, edu in enumerate(educations):
            for m_idx, emp in enumerate(employments):
                for c_idx, sector in enumerate(sectors):
                    pop = int(result_rounded[s_idx, a5_idx, e_idx, m_idx, c_idx])
                    if pop > 0:  # Only save non-zero populations
                        rows.append(
                            {
                                "sex": sex,
                                "age": age,
                                "education": edu,
                                "employment": emp,
                                "sector": sector,
                                "population": pop,
                            }
                        )

synthetic_pop = pd.DataFrame(rows)
output_file = "output/synthetic_population_output.csv"
synthetic_pop.to_csv(output_file, index=False)
print(f"\nSynthetic population saved to {output_file}")
print(f"Total records: {len(synthetic_pop)}")
print(f"Total population: {synthetic_pop['population'].sum()}")
print("\nFirst few rows:")
print(synthetic_pop.head(10))

# 2 Active population attributes

In [ ]:
# Assign professional status and industry to EMP population using HC05_6 + HC05_7
hc05_6 = pd.read_csv("input_data/TF_CENSUS_2021_HC05_6_cleaned.csv")
hc05_7 = pd.read_csv("input_data/TF_CENSUS_2021_HC05_7_cleaned.csv")

# Load base synthetic EMP population
emp_base = synthetic_pop.loc[synthetic_pop["employment"] == "EMP"].copy()

In [ ]:
# Helpers
def parse_age_lower(age_label):
    """Extract lower bound from labels like '25-29' or '100+'."""
    if pd.isna(age_label):
        return np.nan
    s = str(age_label).strip()
    nums = re.findall(r"\d+", s)
    if not nums:
        return np.nan
    return int(nums[0])


def build_lookup(df, key_cols):
    lookup = {}
    for keys, group in df.groupby(key_cols):
        lookup[keys] = group.reset_index(drop=True)
    return lookup


def normalize_probs(values):
    total = values.sum()
    if total <= 0:
        return np.ones(len(values)) / len(values)
    return values / total


def map_age5_to_age15(age5_label):
    """Map 5-year age classes to 15-year classes: 0-14, 15-29, ..., 100+."""
    lo = parse_age_lower(age5_label)
    if pd.isna(lo):
        return None
    lo = int(lo)
    if lo >= 100:
        return "100+"
    start = (lo // 15) * 15
    end = start + 14
    return f"{start}-{end}"


def fit_industry_probs_by_sex(base_gae, industry_probs, target_si, industry_col, max_iter=200, tol=1e-6):
    """
    For each sex, fit a (age15, edu) x industry matrix so that:
      - row sums = synthetic counts by (sex, age15, edu)
      - column sums = target counts by (sex, industry)
    Seed comes from HC05_7 conditional probabilities.
    """
    fitted = {}
    sex_fallback = {}

    # Priors from HC05_7
    prior_l1 = build_lookup(industry_probs, ["CD_SEX", "CD_AGE", "CD_EDU"])
    prior_sex = build_lookup(industry_probs, ["CD_SEX"])

    sexes = sorted(base_gae["CD_SEX"].unique())
    all_industries = sorted(industry_probs[industry_col].unique())

    for sx in sexes:
        gae_s = base_gae[base_gae["CD_SEX"] == sx].copy().reset_index(drop=True)
        if gae_s.empty:
            continue

        # Industries and targets for this sex
        t_s = target_si[target_si["CD_SEX"] == sx][[industry_col, "target"]].copy()
        industries_s = sorted(set(all_industries).union(set(t_s[industry_col].unique())))
        ind_idx = {ind: i for i, ind in enumerate(industries_s)}

        row_tot = gae_s["base_pop"].to_numpy(dtype=float)
        n_rows = len(gae_s)
        n_ind = len(industries_s)
        seed = np.zeros((n_rows, n_ind), dtype=float)

        # Build seed counts per row from HC05_7 conditionals
        for r_idx, r in gae_s.iterrows():
            subset = prior_l1.get((sx, r["CD_AGE"], r["CD_EDU"]))
            if subset is None or subset.empty:
                subset = prior_sex.get(sx)
            if subset is None or subset.empty:
                probs = np.ones(n_ind) / n_ind
            else:
                probs = np.zeros(n_ind, dtype=float)
                for _, rr in subset.iterrows():
                    if rr[industry_col] in ind_idx:
                        probs[ind_idx[rr[industry_col]]] = rr["prob"]
                probs = normalize_probs(probs)
            seed[r_idx, :] = probs * row_tot[r_idx]

        col_target = np.zeros(n_ind, dtype=float)
        for _, tr in t_s.iterrows():
            if tr[industry_col] in ind_idx:
                col_target[ind_idx[tr[industry_col]]] = tr["target"]

        # If some industries have no explicit target, keep seed column totals for them
        seed_col = seed.sum(axis=0)
        missing_target = col_target <= 0
        col_target[missing_target] = seed_col[missing_target]

        # Ensure totals match row totals
        row_total = row_tot.sum()
        col_total = col_target.sum()
        if col_total > 0:
            col_target = col_target * (row_total / col_total)
        else:
            col_target = seed_col * (row_total / max(seed_col.sum(), 1.0))

        # IPF/raking on 2D matrix
        X = seed.copy() + 1e-12
        for _ in range(max_iter):
            # Match rows
            row_sum = X.sum(axis=1)
            row_factor = np.where(row_sum > 0, row_tot / row_sum, 1.0)
            X = (X.T * row_factor).T

            # Match columns
            col_sum = X.sum(axis=0)
            col_factor = np.where(col_sum > 0, col_target / col_sum, 1.0)
            X = X * col_factor

            # Convergence
            row_err = np.max(np.abs(X.sum(axis=1) - row_tot)) if len(row_tot) else 0.0
            col_err = np.max(np.abs(X.sum(axis=0) - col_target)) if len(col_target) else 0.0
            if max(row_err, col_err) < tol:
                break

        # Store fitted conditional probs per (sex, age15, edu)
        row_prob = np.where(
            X.sum(axis=1, keepdims=True) > 0,
            X / X.sum(axis=1, keepdims=True),
            np.ones_like(X) / n_ind,
        )

        for r_idx, r in gae_s.iterrows():
            subset_df = pd.DataFrame({
                "CD_SEX": sx,
                "CD_AGE": r["CD_AGE"],
                "CD_EDU": r["CD_EDU"],
                industry_col: industries_s,
                "prob": row_prob[r_idx, :],
            })
            fitted[(sx, r["CD_AGE"], r["CD_EDU"])] = subset_df

        sex_fallback[sx] = pd.DataFrame({
            "CD_SEX": sx,
            industry_col: industries_s,
            "prob": normalize_probs(col_target),
        })

    return fitted, sex_fallback

## 1.1 Assign industry

In [ ]:
# Column names in HC05 tables
industry_col = "CD_INDUSTRY"
status_col = "CD_PROF_STATUS"

# Map EMP age (5y) to HC05_7 age (15y)
emp_base["age_15"] = emp_base["age"].apply(map_age5_to_age15)

# Assign industry using HC05_7 (sex, age15, education -> industry)
industry_probs = (
    hc05_7.groupby(["CD_SEX", "CD_AGE", "CD_EDU", industry_col], as_index=False)["MS_POP"]
    .sum()
 )
industry_probs["prob"] = industry_probs.groupby(["CD_SEX", "CD_AGE", "CD_EDU"])["MS_POP"].transform(
    lambda x: x / x.sum()
 )

# Build synthetic margins by sex-age15-education
base_gae = (
    emp_base.groupby(["sex", "age_15", "education"], as_index=False)["population"]
    .sum()
    .rename(columns={
        "sex": "CD_SEX",
        "age_15": "CD_AGE",
        "education": "CD_EDU",
        "population": "base_pop",
    })
 )

# Targets for sex-industry from HC05_6, scaled to synthetic sex totals
target_si = hc05_6.groupby(["CD_SEX", industry_col], as_index=False)["MS_POP"].sum().rename(columns={"MS_POP": "target_raw"})

sex_synth = emp_base.groupby("sex", as_index=False)["population"].sum().rename(
    columns={"sex": "CD_SEX", "population": "synth_sex_total"}
 )
sex_target = target_si.groupby("CD_SEX", as_index=False)["target_raw"].sum().rename(
    columns={"target_raw": "target_sex_total"}
 )

target_si = target_si.merge(sex_synth, on="CD_SEX", how="left").merge(sex_target, on="CD_SEX", how="left")
target_si["target"] = np.where(
    target_si["target_sex_total"] > 0,
    target_si["target_raw"] / target_si["target_sex_total"] * target_si["synth_sex_total"],
    0.0,
 )

# Fit calibrated conditionals that satisfy both margins
fitted_l1, fitted_sex = fit_industry_probs_by_sex(
    base_gae=base_gae,
    industry_probs=industry_probs,
    target_si=target_si[["CD_SEX", industry_col, "target"]],
    industry_col=industry_col,
 )

ind_global = industry_probs.copy().reset_index(drop=True)

def get_industry_subset(sex, age15, edu):
    subset = fitted_l1.get((sex, age15, edu))
    if subset is None or subset.empty:
        subset = fitted_sex.get(sex)
    if subset is None or subset.empty:
        subset = ind_global[ind_global["CD_SEX"] == sex]
    if subset is None or subset.empty:
        subset = ind_global
    return subset

emp_with_industry_rows = []

for row in emp_base.itertuples(index=False):
    n = int(row.population)
    if n <= 0:
        continue

    subset = get_industry_subset(row.sex, row.age_15, row.education)
    pvec = normalize_probs(subset["prob"].to_numpy(dtype=float))
    split_counts = rng.multinomial(n, pvec)

    for idx, count in enumerate(split_counts):
        if count == 0:
            continue
        emp_with_industry_rows.append({
            "sex": row.sex,
            "age": row.age,
            "education": row.education,
            "sector": row.sector,
            "industry": subset.iloc[idx][industry_col],
            "population": int(count),
        })

emp_with_industry = pd.DataFrame(emp_with_industry_rows)

## 1.2 Professional status

In [ ]:
# Assign professional status using HC05_6 (sex, age5, industry -> status)
status_probs = (
    hc05_6.groupby(["CD_SEX", "CD_AGE", industry_col, status_col], as_index=False)["MS_POP"]
    .sum()
 )
status_probs["prob"] = status_probs.groupby(["CD_SEX", "CD_AGE", industry_col])["MS_POP"].transform(
    lambda x: x / x.sum()
 )

st_l1 = build_lookup(status_probs, ["CD_SEX", "CD_AGE", industry_col])
st_l2 = build_lookup(status_probs, ["CD_SEX", "CD_AGE"])
st_l3 = build_lookup(status_probs, ["CD_SEX", industry_col])
st_l4 = build_lookup(status_probs, ["CD_SEX"])
st_global = status_probs.copy().reset_index(drop=True)

def get_status_subset(sex, age5, industry):
    subset = st_l1.get((sex, age5, industry))
    if subset is None or subset.empty:
        subset = st_l2.get((sex, age5))
    if subset is None or subset.empty:
        subset = st_l3.get((sex, industry))
    if subset is None or subset.empty:
        subset = st_l4.get(sex)
    if subset is None or subset.empty:
        subset = st_global
    return subset

emp_enriched_rows = []

for row in emp_with_industry.itertuples(index=False):
    n = int(row.population)
    if n <= 0:
        continue

    subset = get_status_subset(row.sex, row.age, row.industry)
    pvec = normalize_probs(subset["prob"].to_numpy(dtype=float))
    split_counts = rng.multinomial(n, pvec)

    for idx, count in enumerate(split_counts):
        if count == 0:
            continue
        emp_enriched_rows.append({
            "sex": row.sex,
            "age": row.age,
            "education": row.education,
            "sector": row.sector,
            "industry": row.industry,
            "professional_status": subset.iloc[idx][status_col],
            "population": int(count),
        })

emp_enriched = pd.DataFrame(emp_enriched_rows)

print("EMP base population:", int(emp_base["population"].sum()))
print("EMP with industry population:", int(emp_with_industry["population"].sum()))
print("EMP enriched population:", int(emp_enriched["population"].sum()))

# Save the data to CSV file
output_file = "output/employed_population.csv"
emp_enriched.to_csv(output_file, index=False)